In [ ]:
from pqc_iot_simulator.pqc_iot_sim.components import Network, Logger
from pqc_iot_simulator.pqc_iot_sim.engines import MininetWiFiEngine

import pandas as pd
from pathlib import Path

Logger.activate()

In [ ]:
payload = {
    "sensor": "temperature",
    "value": 28.5,
    "unit": "celsius",
    "location": "lab"
}

output_dir = Path("outputs/notebook_demo")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def criar_rede_e_engine(crypto_mode="classical"):
    rede = Network(verbose=True)

    rede.set_ready_topology("grade", 3, 3)
    rede.set_protocol("mqtt")
    rede.set_crypto_mode(crypto_mode)

    engine = MininetWiFiEngine(
        network=rede,
        logger=rede.logger,
        default_bw=10,
        default_delay="5ms",
        default_loss=0,
        link_mode="infrastructure",
        station_range=200,
        ap_range=200,
        association_wait_seconds=1.0
    )

    engine.build()
    engine.start()

    return rede, engine

In [ ]:
rede, engine = criar_rede_e_engine("classical")

link_metrics = engine.collect_link_metrics(
    source="iot_1",
    destination="server_1"
)

send_result = rede.send(
    source="iot_1",
    destination="server_1",
    payload=payload,
    link_metrics=link_metrics
)

summary = rede.metrics()

summary

In [ ]:
pd.DataFrame(rede.transmissions())

In [ ]:
print(engine.ping("iot_1", "server_1"))

In [ ]:
rede.export_metrics_json("outputs/notebook_demo/metrics_classical.json")
rede.export_metrics_csv("outputs/notebook_demo/metrics_classical.csv")
engine.export_mapping_json("outputs/notebook_demo/mapping_classical.json")

In [ ]:
engine.stop()